# Weekly snapshot — projects, tasks, allocations

Captures the full state of three live tables — `projects`, `tasks`, and `allocations` — into a separate `datateam_snapshots` service every week, tagged with the Monday-anchored date of the week being snapshotted.

**Why this matters.** The team's convention for allocations is that current/future-week rows are *estimates* (what someone thinks they'll work on a project), and past-week rows get edited later to reflect *actuals*. Without this job, the original estimate is overwritten and unrecoverable — making estimate-vs-actual calibration impossible. The Sunday-night / Monday-morning timing is deliberate: capture each week's rows while they're still in their estimate state, before reflection-and-update.

**Schedule.** Once per week, late Sunday or very early Monday. Idempotent — re-running for the same week is a safety no-op.

**Self-bootstrapping.** First run creates the target service and all three tables. Subsequent runs auto-add any new source fields to the snapshot tables (forward-only — fields are never removed once added).

**Auto-tracking renames.** Each source's `CreationDate / Creator / EditDate / Editor` get renamed to `source_creation_date / source_creator / source_edit_date / source_editor` so the snapshot row's own AGOL-managed editor-tracking fields don't collide.

**Recycle-bin tolerance.** The find-or-create logic filters out items that appear in search but can't actually be hydrated (e.g. still in AGOL's recycle bin) — a stale search result can't trip up a fresh run.

**Sources of truth.** `datateam_portfolio_v2` (projects layer 0, tasks layer 1), `datateam_capacity_v2` (allocations layer 3).

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, FeatureLayerCollection
import datetime as dt

gis = GIS("home")
print(f"Authenticated as: {gis.users.me.username} @ {gis.url}")

## Config

Sources, target service, table names, and the schema for the two extra columns we add to every snapshot row. Set `KNOWN_SNAPSHOT_ITEM_ID` after the first successful run so future runs skip the search step.

In [ ]:
SOURCE_PROJECTS_URL    = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/0"
SOURCE_TASKS_URL       = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"
SOURCE_ALLOCATIONS_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_capacity_v2/FeatureServer/3"

SNAPSHOT_SERVICE_NAME = "datateam_snapshots"
# Item ID of the snapshot service. Leave None on the first-ever run; update
# this after Phase 2 prints the id so future runs skip the search step.
KNOWN_SNAPSHOT_ITEM_ID = None

# Snapshot tables. Names mirror the sources.
PROJECTS_TABLE    = "projects_snapshots"
TASKS_TABLE       = "tasks_snapshots"
ALLOCATIONS_TABLE = "allocations_snapshots"

# Fields that AGOL auto-manages on every layer (we can't write these on the
# snapshot row, so we rename them on copy to preserve the source's values).
AUTO_TRACKING_RENAMES = {
    "CreationDate": "source_creation_date",
    "Creator":      "source_creator",
    "EditDate":     "source_edit_date",
    "Editor":       "source_editor",
}
# Fields we drop on copy (the snapshot row has its own).
DROP_ON_COPY = {"ObjectId", "GlobalID"}

# Extra metadata columns added to every snapshot row.
# snapshot_week is esriFieldTypeDateOnly — written as an ISO date string
# "YYYY-MM-DD" and queried with DATE 'YYYY-MM-DD' literals. Do NOT write
# milliseconds-since-epoch to a DateOnly field; AGOL rejects it.
EXTRA_FIELDS = [
    {"name": "snapshot_week", "type": "esriFieldTypeDateOnly", "alias": "Snapshot Week",
     "sqlType": "sqlTypeOther", "nullable": False, "editable": True,
     "description": '{"value": "Monday-anchored date identifying which week this snapshot represents (YYYY-MM-DD).", "fieldValueType": ""}'},
    {"name": "source_object_id", "type": "esriFieldTypeInteger", "alias": "Source ObjectId",
     "sqlType": "sqlTypeInteger", "nullable": True, "editable": True,
     "description": '{"value": "The ObjectId of the original row in the live source layer at snapshot time.", "fieldValueType": ""}'},
]

## Phase 1 — Compute this run's snapshot week

Monday of the current week. Python's `weekday()` returns 0=Monday..6=Sunday. Matches the app's `RESOURCES_DATA.weeks` Monday convention.

In [ ]:
today = dt.date.today()
monday = today - dt.timedelta(days=today.weekday())
SNAPSHOT_WEEK = monday.isoformat()  # e.g. '2026-05-18' — ISO date string for writes & queries
print(f"Today is {today.isoformat()} ({today.strftime('%A')})")
print(f"Snapshot week (Monday-anchored): {SNAPSHOT_WEEK}")

## Phase 2 — Find or create the snapshot service

Self-bootstrapping. Creates the service on first run, finds it on subsequent runs.

**Recycle-bin tolerance.** Items deleted via the AGOL UI go to a recycle bin and remain searchable. Their cached fields (id, title, url) look normal, but a fresh REST hydration fails. The `_is_accessible(item)` helper forces a hydration call so we can detect and skip those stale entries — otherwise the search would happily return a tombstone.

In [ ]:
def _is_accessible(item):
    """Force a REST hydration to verify the item actually exists. Items in
    AGOL's recycle bin appear in search results with cached fields (.url,
    .title) already populated, so simple attribute access succeeds — but
    a real hydration call fails. That's what we have to catch."""
    try:
        item._hydrate()
        return True
    except Exception:
        return False

def get_or_create_snapshot_service(name: str):
    if KNOWN_SNAPSHOT_ITEM_ID:
        item = gis.content.get(KNOWN_SNAPSHOT_ITEM_ID)
        if item and _is_accessible(item) and item.type == "Feature Service":
            print(f"Snapshot service found by ID: {item.id} ({item.title})")
            return item
    me = gis.users.me.username
    hits = [i for i in gis.content.search(f'title:"{name}" owner:{me}', item_type="Feature Service")
            if i.title == name and _is_accessible(i)]
    if hits:
        print(f"Snapshot service exists (owned by me): {hits[0].id} ({hits[0].title})")
        return hits[0]
    target_url_substr = f"/{name}/FeatureServer"
    for i in gis.content.search(name, item_type="Feature Service", max_items=50):
        if not _is_accessible(i):
            continue
        if target_url_substr in (i.url or ""):
            print(f"Snapshot service found by URL: {i.id} ({i.title}, owner={i.owner})")
            return i
    print(f"Creating new snapshot service: {name}")
    item = gis.content.create_service(
        name=name,
        service_description="Weekly snapshots of datateam_portfolio_v2 (projects, tasks) and datateam_capacity_v2 (allocations). Written by snapshot_weekly.ipynb. Read-only otherwise.",
        has_static_data=False,
        max_record_count=4000,
        capabilities="Query",
        service_type="featureService",
    )
    print(f"Created: {item.id}")
    return item

snap_item = get_or_create_snapshot_service(SNAPSHOT_SERVICE_NAME)
snap_flc  = FeatureLayerCollection.fromitem(snap_item)
print(f"\nIf this is the first run, update KNOWN_SNAPSHOT_ITEM_ID in the Config cell to {snap_item.id!r} for faster subsequent runs.")

## Phase 3 — Ensure each snapshot table has the current source schema

For each of the three sources:
- If the snapshot table is missing → create it from the source schema (renaming auto-tracking fields, adding `snapshot_week` + `source_object_id`).
- If it exists but missing fields → add the missing ones in place (forward-only; this notebook never removes or changes field types of existing fields).

In [ ]:
def build_snapshot_table_def(src_layer, target_name):
    src_props = dict(src_layer.properties)
    src_fields = src_props.get("fields", [])
    snap_fields = []
    for f in src_fields:
        nm = f["name"]
        if nm in DROP_ON_COPY:
            continue
        nf = dict(f)
        if nm in AUTO_TRACKING_RENAMES:
            new_name = AUTO_TRACKING_RENAMES[nm]
            nf["name"] = new_name
            nf["alias"] = new_name.replace("_", " ").title()
            nf.pop("defaultValue", None)
        snap_fields.extend([nf])
    snap_fields.extend(EXTRA_FIELDS)
    return {
        "name": target_name,
        "type": "Table",
        "displayField": "project_number" if "project_number" in [x["name"] for x in snap_fields] else snap_fields[0]["name"],
        "description": f"Weekly snapshot of {src_props.get('name')}.",
        "objectIdField": "ObjectId",
        "fields": snap_fields,
        "capabilities": "Create,Query",
        "supportsAdvancedQueries": True,
        "hasAttachments": False,
        "indexes": [
            {"name": f"{target_name}_snapshot_week_idx", "fields": "snapshot_week", "isAscending": True, "isUnique": False,
             "description": "Speeds up 'all snapshots for week X' queries."},
        ],
    }

def ensure_snapshot_table(item_id: str, src_layer, target_name: str):
    flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
    existing = next((t for t in flc.tables if t.properties.name == target_name), None)
    if existing:
        existing_names = {f["name"] for f in existing.properties.fields}
        expected_names = {f["name"] for f in build_snapshot_table_def(src_layer, target_name)["fields"]} | {"ObjectId"}
        missing = expected_names - existing_names
        if missing:
            wanted_defs = {f["name"]: f for f in build_snapshot_table_def(src_layer, target_name)["fields"]}
            to_add = [wanted_defs[n] for n in sorted(missing) if n in wanted_defs]
            if to_add:
                print(f"  Adding to '{target_name}': {[f['name'] for f in to_add]}")
                existing.manager.add_to_definition({"fields": to_add})
        else:
            print(f"  Table '{target_name}' schema is up to date ({len(existing.properties.fields)} fields)")
        return existing
    tdef = build_snapshot_table_def(src_layer, target_name)
    print(f"  Creating table '{target_name}' ({len(tdef['fields'])} fields)")
    flc.manager.add_to_definition({"tables": [tdef]})
    flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
    return next(t for t in flc.tables if t.properties.name == target_name)

src_projects = FeatureLayer(SOURCE_PROJECTS_URL,    gis=gis)
src_tasks    = FeatureLayer(SOURCE_TASKS_URL,       gis=gis)
src_allocs   = FeatureLayer(SOURCE_ALLOCATIONS_URL, gis=gis)
print(f"Source: {src_projects.query(where='1=1', return_count_only=True)} projects, "
      f"{src_tasks.query(where='1=1', return_count_only=True)} tasks, "
      f"{src_allocs.query(where='1=1', return_count_only=True)} allocations")

print("\nEnsuring snapshot schema...")
tgt_projects = ensure_snapshot_table(snap_item.id, src_projects, PROJECTS_TABLE)
tgt_tasks    = ensure_snapshot_table(snap_item.id, src_tasks,    TASKS_TABLE)
tgt_allocs   = ensure_snapshot_table(snap_item.id, src_allocs,   ALLOCATIONS_TABLE)

## Phase 4 — Idempotency check

If a snapshot already exists for this week, stop. Re-running on the same week is a safety no-op (per-table, so if one table failed before the others were written, only the missing ones get filled).

To intentionally re-snapshot a week (e.g. data was corrected after a snapshot ran), use the cleanup snippet at the bottom of this notebook first to delete the existing rows for that week.

In [ ]:
def week_already_snapshotted(tgt_layer):
    # snapshot_week is esriFieldTypeDateOnly — query with DATE 'YYYY-MM-DD' literal.
    where = f"snapshot_week = DATE '{SNAPSHOT_WEEK}'"
    c = tgt_layer.query(where=where, return_count_only=True)
    return c > 0

proj_done  = week_already_snapshotted(tgt_projects)
task_done  = week_already_snapshotted(tgt_tasks)
alloc_done = week_already_snapshotted(tgt_allocs)

if proj_done and task_done and alloc_done:
    print(f"Snapshot for week {SNAPSHOT_WEEK} already exists in all tables. Nothing to do.")
    print("To force a re-snapshot, run the cleanup cell at the bottom and then re-run this cell forward.")
    SKIP_WRITE = True
else:
    SKIP_WRITE = False
    print(f"Will snapshot week {SNAPSHOT_WEEK}: projects_done={proj_done}, tasks_done={task_done}, alloc_done={alloc_done}")

## Phase 5 — Write the snapshots

Pulls every row from each source layer (live + soft-deleted), transforms each attribute set (rename auto-tracking, drop ObjectId, add `snapshot_week` + `source_object_id`), and appends in 1,000-row chunks.

In [ ]:
def transform_row(src_attrs):
    out = {}
    for k, v in src_attrs.items():
        if k in DROP_ON_COPY:
            if k == "ObjectId":
                out["source_object_id"] = v
            continue
        if k in AUTO_TRACKING_RENAMES:
            out[AUTO_TRACKING_RENAMES[k]] = v
        else:
            out[k] = v
    # snapshot_week is esriFieldTypeDateOnly — write as ISO date string "YYYY-MM-DD".
    out["snapshot_week"] = SNAPSHOT_WEEK
    return out

def snapshot_one(src_layer, tgt_layer, label):
    # 1=1 picks up everything including soft-deleted rows.
    fset = src_layer.query(where="1=1", out_fields="*", return_geometry=False)
    raw = fset.features
    transformed = [{"attributes": transform_row(f.attributes)} for f in raw]
    added = 0
    for i in range(0, len(transformed), 1000):
        chunk = transformed[i:i+1000]
        result = tgt_layer.edit_features(adds=chunk)
        ok = sum(1 for r in result.get("addResults", []) if r.get("success"))
        added += ok
        if ok != len(chunk):
            fails = [r for r in result.get("addResults", []) if not r.get("success")]
            print(f"  ! {label}: only {ok}/{len(chunk)} added (chunk starting {i})")
            for fr in fails[:3]:
                print(f"    fail sample: {fr}")
    print(f"  {label}: {added}/{len(transformed)} rows snapshotted for week {SNAPSHOT_WEEK}")
    return added

if SKIP_WRITE:
    print("Skipped (already snapshotted this week).")
else:
    print("Writing snapshots...")
    if not proj_done:
        snapshot_one(src_projects, tgt_projects, "projects")
    if not task_done:
        snapshot_one(src_tasks,    tgt_tasks,    "tasks")
    if not alloc_done:
        snapshot_one(src_allocs,   tgt_allocs,   "allocations")

## Phase 6 — Verification

Prints row counts per `snapshot_week` for each table so you can see history accumulating.

In [ ]:
def list_snapshots(tgt_layer, label):
    out_stats = [{"statisticType": "count", "onStatisticField": "snapshot_week", "outStatisticFieldName": "n"}]
    res = tgt_layer.query(where="1=1", out_statistics=out_stats, group_by_fields_for_statistics="snapshot_week", order_by_fields="snapshot_week ASC")
    print(f"{label} snapshot counts per week:")
    for f in res.features:
        wk = f.attributes.get("snapshot_week")
        if wk is None:
            wk_str = "(null)"
        elif isinstance(wk, (int, float)):
            # Defensive: some AGOL configurations return DateOnly as ms-since-epoch.
            wk_str = dt.datetime.fromtimestamp(wk / 1000, tz=dt.timezone.utc).date().isoformat()
        else:
            wk_str = str(wk)
        print(f"  {wk_str}: {f.attributes['n']} rows")

list_snapshots(tgt_projects, "projects_snapshots")
print()
list_snapshots(tgt_tasks,    "tasks_snapshots")
print()
list_snapshots(tgt_allocs,   "allocations_snapshots")
print()
print(f"Snapshot service item: {snap_item.id}")
print("Per-table URLs:")
flc = FeatureLayerCollection.fromitem(gis.content.get(snap_item.id))
for t in flc.tables:
    print(f"  {t.properties.name:24s}  {t.url}")

## Optional — Delete snapshots for a specific week

Use this if you need to re-snapshot a week (e.g. source data was corrected after the snapshot ran, and you want the corrected version stored). Uncomment, set `WEEK_TO_DELETE`, and run. Idempotent — safe to re-run if a week has no rows.

This is the right tool for any future recovery. **Do not delete the snapshot service itself** — once it's running, just truncate the affected week(s) and re-run the snapshot.

In [ ]:
# WEEK_TO_DELETE = '2026-05-11'
# for tgt in [tgt_projects, tgt_tasks, tgt_allocs]:
#     oid_field = tgt.properties.objectIdField  # whatever AGOL named it ('ObjectId' / 'OBJECTID' / etc.)
#     where = f"snapshot_week = DATE '{WEEK_TO_DELETE}'"
#     ids = [f.attributes[oid_field] for f in tgt.query(where=where, out_fields=oid_field, return_geometry=False).features]
#     if not ids:
#         print(f"{tgt.properties.name}: no rows for {WEEK_TO_DELETE}")
#         continue
#     r = tgt.edit_features(deletes=ids)
#     ok = sum(1 for x in r.get('deleteResults', []) if x.get('success'))
#     print(f"{tgt.properties.name}: deleted {ok}/{len(ids)} rows for {WEEK_TO_DELETE}")